# Notebook 04 — Linear Algebra

**numpy-mastery** · Module 04 of 06

> **Goal**: master matrix multiplication, norms, determinants, inverses,
> and the core `np.linalg` toolkit that underlies every ML model.

**What you'll be able to do after this notebook:**
- Distinguish element-wise multiplication from matrix multiplication
- Use `np.linalg` to solve linear systems, invert matrices, and compute norms
- Understand eigenvalues/eigenvectors and the SVD at a working level
- Implement linear regression from the normal equation

---


## 0 · Setup

In [1]:
import numpy as np
from numpy import linalg as LA

---
## 1 · Matrix multiplication vs element-wise multiplication

This is the single most important distinction in this notebook.
Confusing `*` and `@` is one of the most common bugs in ML code.


In [5]:
A = np.array([[1, 2],
              [3, 4]])
B = np.array([[5, 6],
              [7, 8]])

# Element-wise (Hadmard) product - same shape required
print("A * B (element-wise):")
print(A * B)
# [[1*5, 2*6],     [[5  12]
#  [3*7, 4*8]]  =   [21 32]]

print("\nA @ B (matrix multiply):")
print(A @ B)
# row·col dot products:
# [[1*5+2*7, 1*6+2*8],   [[19 22]
#  [3*5+4*7, 3*6+4*8]] =  [43 50]]

A * B (element-wise):
[[ 5 12]
 [21 32]]

A @ B (matrix multiply):
[[19 22]
 [43 50]]


### Three equivalent ways to matrix-multiply

`np.dot`, `np.matmul`, and the `@` operator all do the same thing for 2-D
arrays. `@` is the modern, preferred syntax (Python 3.5+).


In [6]:
A = np.array([[1, 2], [3, 4]])
B = np.array([[5, 6], [7, 8]])


print(np.dot(A, B))
print(np.matmul(A, B))
print(A @ B)

[[19 22]
 [43 50]]
[[19 22]
 [43 50]]
[[19 22]
 [43 50]]


### Matrix–vector multiplication

A matrix times a vector follows the same rule: each output element is the
dot product of a matrix row with the vector.


In [8]:
M = np.array([[1, 2, 3],
              [4, 5, 6]])   # shape (2, 3)
v = np.array([1, 0, 1])     # shape (3,)

result = M @ v
print(result)
# [1*1+2*0+3*1, 4*1+5*0+6*1] = [4, 10]

[ 4 10]


### Shape rule for matrix multiplication

For `A @ B` to work: `A.shape = (m, n)` and `B.shape = (n, p)` →
result shape is `(m, p)`. **The inner dimensions must match.**

In [11]:
A = np.ones((3, 4))   # (3, 4)
B = np.ones((4, 5))   # (4, 5)
C = A @ B             # (3, 5)  — inner dims (4) match, they cancel out
print("C shape:", C.shape)

try:
    A @ np.ones((3, 5))   # (3,4) @ (3,5) — inner dims 4 != 3, fails
except ValueError as e:
    print("Error:", e)

C shape: (3, 5)
Error: matmul: Input operand 1 has a mismatch in its core dimension 0, with gufunc signature (n?,k),(k,m?)->(n?,m?) (size 3 is different from 4)


---
## 2 · Transpose

Flips rows and columns: `M.T[i, j] == M[j, i]`. Frequently needed to make
shapes compatible for matrix multiplication.

In [ ]:
M = np.array([[1, 2, 3],
              [4, 5, 6]])   # shape (2, 3)

print(M.T)
print(M.T.shape)

[[1 4]
 [2 5]
 [3 6]]
(3, 2)


In [13]:
# Common pattern: X @ X.T  or  X.T @ X  (covariance-like computations)
X = np.array([[1.0, 2.0], [3.0, 4.0], [5.0, 6.0]])  # (3, 2)
print("\nX.T @ X shape:", (X.T @ X).shape)   # (2,3)@(3,2) -> (2,2)
print("X @ X.T shape:", (X @ X.T).shape)     # (3,2)@(2,3) -> (3,3)


X.T @ X shape: (2, 2)
X @ X.T shape: (3, 3)


---
## 3 · Determinant, inverse, trace, rank

| Function | Meaning |
|----------|---------|
| `LA.det(A)` | scalar — measures how A scales volume; `0` means A is singular (non-invertible) |
| `LA.inv(A)` | matrix `A⁻¹` such that `A @ A⁻¹ = I` |
| `np.trace(A)` | sum of diagonal elements |
| `LA.matrix_rank(A)` | number of linearly independent rows/columns |


In [16]:
A = np.array([[4.0, 7.0],
              [2.0, 6.0]])

det_A = LA.det(A)
print("det(A)", det_A)

A_inv = LA.inv(A)
print("\n A inverse:\n", A_inv)

# Verify: A @ A_inv should be the identity matrix
print("\nA @ A_inv:\n", A @ A_inv)
print("Is identity:", np.allclose(A @ A_inv, np.eye(2)))

print("\ntrace(A):", np.trace(A))   # 4 + 6 = 10
print("rank(A) :", LA.matrix_rank(A))  # 2 (full rank — invertible)

det(A) 10.000000000000002

 A inverse:
 [[ 0.6 -0.7]
 [-0.2  0.4]]

A @ A_inv:
 [[ 1.00000000e+00 -1.11022302e-16]
 [ 1.11022302e-16  1.00000000e+00]]
Is identity: True

trace(A): 10.0
rank(A) : 2


### A singular matrix (non-invertible)

When `det(A) == 0`, the matrix cannot be inverted — `LA.inv` will raise an error.

In [17]:
singular = np.array([[1.0, 2.0],
                      [2.0, 4.0]])   # row 2 = 2 * row 1 → linearly dependent

print("det:", LA.det(singular))        # 0.0
print("rank:", LA.matrix_rank(singular))  # 1, not 2 — not full rank

try:
    LA.inv(singular)
except LA.LinAlgError as e:
    print("Error:", e)

det: 0.0
rank: 1
Error: Singular matrix


---
## 4 · Norms — measuring "size" of vectors and matrices

A norm is a single number describing the magnitude of a vector or matrix.

In [18]:
v = np.array([3.0, 4.0])

print("L2 norm (default, Euclidean):", LA.norm(v))     # sqrt(3²+4²) = 5.0
print("L1 norm (Manhattan):", LA.norm(v, ord=1))         # |3|+|4| = 7.0
print("L∞ norm (max abs):", LA.norm(v, ord=np.inf))      # max(|3|,|4|) = 4.0

L2 norm (default, Euclidean): 5.0
L1 norm (Manhattan): 7.0
L∞ norm (max abs): 4.0


### Matrix norms and the Frobenius norm

For matrices, the default norm is the **Frobenius norm**: treat every
element as part of one giant vector and take its L2 norm.

In [19]:
M = np.array([[1.0, 2.0],
              [3.0, 4.0]])

frob = LA.norm(M)          # sqrt(1² + 2² + 3² + 4²) = sqrt(30)
print("Frobenius norm:", frob)
print("Manual check  :", np.sqrt((M**2).sum()))

Frobenius norm: 5.477225575051661
Manual check  : 5.477225575051661


### Computing distances between points using norms

A common pattern: the Euclidean distance between two points is the
L2 norm of their difference.

In [20]:
p1 = np.array([1.0, 2.0])
p2 = np.array([4.0, 6.0])

distance = LA.norm(p1 - p2)
print("distance:", distance)   # sqrt((4-1)² + (6-2)²) = sqrt(9+16) = 5.0

distance: 5.0


---
## 5 · Solving linear systems — `LA.solve`

Given `Ax = b`, solve for `x`. **Always prefer `LA.solve` over computing
`LA.inv(A) @ b`** — it's faster and numerically more stable.

### Numerical example

System: `3x + y = 9` and `x + 2y = 8`. By substitution, the answer is `x=2, y=3`.


In [21]:
A = np.array([[3.0, 1.0],
              [1.0, 2.0]])
b = np.array([9.0, 8.0])

x = LA.solve(A, b)
print("solution:", x)   # [2. 3.]

# Verify: A @ x should equal b
print("A @ x:", A @ x)
print("matches b:", np.allclose(A @ x, b))


solution: [2. 3.]
A @ x: [9. 8.]
matches b: True


### Why not just use the inverse?

`LA.inv(A) @ b` gives the same answer here, but `LA.solve` avoids
explicitly computing the inverse, which is slower and accumulates more
floating-point error — especially on larger or ill-conditioned matrices.

In [22]:
# Both work for this small example, but solve is the professional standard
x_via_inv   = LA.inv(A) @ b
x_via_solve = LA.solve(A, b)
print("via inv  :", x_via_inv)
print("via solve:", x_via_solve)

via inv  : [2. 3.]
via solve: [2. 3.]


---
## 6 · Eigenvalues and eigenvectors

For a square matrix `A`, an eigenvector `v` and eigenvalue `λ` satisfy:

```
A @ v = λ * v
```

In words: multiplying `A` by `v` only **scales** `v` — it doesn't change
its direction.

In [23]:
A = np.array([[2.0, 1.0],
              [1.0, 3.0]])

eigenvalues, eigenvectors = LA.eig(A)

print("eigenvalues :", eigenvalues)
print("eigenvectors (columns):\n", eigenvectors)

# Verify the defining property for the first eigenvector
v0 = eigenvectors[:, 0]
lam0 = eigenvalues[0]

print("\nA @ v0   :", A @ v0)
print("λ0 * v0  :", lam0 * v0)
print("Match    :", np.allclose(A @ v0, lam0 * v0))


eigenvalues : [1.38196601 3.61803399]
eigenvectors (columns):
 [[-0.85065081 -0.52573111]
 [ 0.52573111 -0.85065081]]

A @ v0   : [-1.1755705   0.72654253]
λ0 * v0  : [-1.1755705   0.72654253]
Match    : True


---
## 7 · Singular Value Decomposition (SVD)

Any matrix `A` (even non-square!) can be decomposed as:

```
A = U @ diag(S) @ Vt
```

where `U` and `Vt` are orthogonal matrices and `S` contains the singular
values (always ≥ 0, sorted descending). SVD underlies PCA, recommender
systems, and dimensionality reduction.

In [24]:
M = np.array([[3.0, 1.0],
              [1.0, 3.0],
              [1.0, 1.0]])    # shape (3, 2) — works even though it's not square!

U, S, Vt = LA.svd(M, full_matrices=False)

print("U shape :", U.shape)    # (3, 2)
print("S       :", S)          # singular values, descending
print("Vt shape:", Vt.shape)   # (2, 2)

# Reconstruct the original matrix
reconstructed = U @ np.diag(S) @ Vt
print("\nReconstruction matches:", np.allclose(M, reconstructed))

U shape : (3, 2)
S       : [4.24264069 2.        ]
Vt shape: (2, 2)

Reconstruction matches: True


---
## 8 · Putting it together — linear regression via the normal equation

Linear regression finds the `θ` that best fits `y ≈ Xθ`.  
The closed-form solution (the **normal equation**) is:

```
θ = (XᵀX)⁻¹ Xᵀy
```

We solve this with `LA.solve` rather than explicitly inverting — exactly
the lesson from Section 5.

In [25]:
# Synthetic data: true relationship is y = 3x + 2 + noise
rng = np.random.default_rng(42)
n = 50
x_raw = rng.uniform(0, 10, n)
y = 3 * x_raw + 2 + rng.normal(0, 1, n)

# Build design matrix with a bias (intercept) column of 1s
X = np.column_stack([np.ones(n), x_raw])   # shape (50, 2)

# Normal equation: θ = (XᵀX)⁻¹ Xᵀy   — solved via LA.solve, not LA.inv
theta = LA.solve(X.T @ X, X.T @ y)

print(f"Intercept (true=2): {theta[0]:.4f}")
print(f"Slope     (true=3): {theta[1]:.4f}")

# R² score — how well the model fits
y_pred  = X @ theta
ss_res  = np.sum((y - y_pred) ** 2)
ss_tot  = np.sum((y - y.mean()) ** 2)
r2      = 1 - ss_res / ss_tot
print(f"R² score: {r2:.4f}")


Intercept (true=2): 1.7687
Slope     (true=3): 3.0121
R² score: 0.9919


---
## 9 · Summary cheatsheet

```python
# Multiplication
A * B          # element-wise (Hadamard) — same shape required
A @ B          # matrix multiply — inner dims must match
np.dot(A, B)   # same as @  for 2-D arrays
M.T            # transpose

# np.linalg (imported as LA)
LA.det(A)            # determinant
LA.inv(A)             # inverse (raises LinAlgError if singular)
LA.matrix_rank(A)     # rank
np.trace(A)           # sum of diagonal

LA.norm(v)            # L2 norm (default)
LA.norm(v, ord=1)     # L1 norm
LA.norm(v, ord=np.inf) # L∞ norm

LA.solve(A, b)        # solve Ax = b  — prefer over inv(A) @ b
LA.eig(A)             # eigenvalues, eigenvectors
LA.svd(A)             # U, S, Vt decomposition
```

---
